# ETFs: a model that prices the cross-section instead of predicting it

The three members before this one share a shape. [`11a_pca`](11a_pca.ipynb),
[`11b_ipca`](11b_ipca.ipynb) and
[`11c_conditional_autoencoder`](11c_conditional_autoencoder.ipynb) all split the problem in two:
estimate what each fund is exposed to, estimate what those exposures earned, multiply. They
differ in how the first half is written down - not at all, a linear map, a network - and in
nothing else.

**The stochastic discount factor does not split it.** It asks for a single series that prices
every fund at once: a weighting of the cross-section whose covariance with each fund's return
accounts for that fund's average return. That is a statement about *no arbitrage* rather than
about forecasting, and it is fitted by driving pricing errors toward zero rather than by
minimising a forecast error. The two networks it trains - one producing the discount-factor
weights from the cross-section, one producing each fund's exposure to it - are trained against
that pricing objective.

A predicted return still comes out, which is why this notebook sits beside the others and is
scored the same way. But the number being optimised is not the number being reported, and that
gap is the thing to hold on to while reading the result.

**It is also the only member that reads macro data.** The other four see the fund panel and the
feature matrix. This one additionally takes eleven macro series as context for the
discount factor, which raises a question none of the others have to answer: what was actually
knowable on the decision date. Section 2 is where that contract is read back out of the resolved
specification rather than asserted here.


<!-- ML4T_ZH_BEGIN c0000-p000 -->
# ETF：為橫斷面定價，而非預測橫斷面的模型

前面三個成員採用相同形式。[`11a_pca`](11a_pca.ipynb)、[`11b_ipca`](11b_ipca.ipynb) 與 [`11c_conditional_autoencoder`](11c_conditional_autoencoder.ipynb) 都將問題拆成兩部分：估計每檔基金的曝險、估計這些曝險帶來的報酬，再將兩者相乘。它們只在第一部分的表示方式上不同：不使用特徵、使用線性映射，或使用神經網路；其他部分相同。

**隨機折現因子不將問題拆開。** 它尋找能同時為所有基金定價的單一序列：對橫斷面加權，使加權序列與每檔基金報酬率的共變異數，可以解釋該基金的平均報酬率。這是關於*無套利*的主張，而非預測；擬合時要將定價誤差推向零，而非最小化預測誤差。模型訓練的兩個神經網路——一個從橫斷面產生折現因子的權重，另一個產生各基金對折現因子的曝險——都以此定價目標訓練。

模型仍會產生預測報酬率，因此本 Notebook 與其他模型並列，並以同樣方式評分。但最佳化的數值並非報告的數值；閱讀結果時，必須記住這項差距。

**這也是唯一讀取總體經濟資料的成員。** 其他四個模型只使用基金面板及特徵矩陣；此模型另外使用十一個總體經濟序列作為折現因子的背景資訊，因此必須回答其他模型不必回答的問題：在決策日實際可得的資訊是什麼？第 2 節會從已解析的規格讀回這項約定，而非只在此聲稱。
<!-- ML4T_ZH_END c0000-p000 -->

**The checkpoint schedule is phased, and all of it is published.** Training runs an
unconditional stage and then a conditional one, and `case_studies/config/sdf/sdf.yaml` declares
saves inside each. Fixing the last of them in advance would be the right instinct about the
trap - picking the checkpoint with the best validation IC is choosing after seeing the answer -
and the wrong remedy, because it discards models the case study paid to train and leaves the
choice to be made again, undocumented, downstream. Publishing the whole schedule lets
[`14_backtest`](14_backtest.ipynb) select on validation backtest Sharpe with every candidate
visible and the rule stated. **Selection happens there, not here.**

**Learning objectives.** By the end of this notebook you will be able to:

- Say what a pricing objective asks for that a forecasting objective does not.
- Read a phased training schedule and work out how many prediction sets a label will owe.
- Read a point-in-time data contract out of a resolved specification and say what it rules out.
- Say why the macro context belongs inside the model's identity rather than beside it.

**Book reference**: Chapter 14, Section 14.7 (The stochastic discount factor and the supervised
autoencoder models). Chapter 6, Section 6.7 (Search accounting and run logging) introduces the
run log this notebook writes to.

**Prerequisites**: [`03_financial_features`](03_financial_features.ipynb) and
[`04_model_based_features`](04_model_based_features.ipynb) for the feature matrix,
[`05_evaluation`](05_evaluation.ipynb) for the walk-forward folds, and
[`11c_conditional_autoencoder`](11c_conditional_autoencoder.ipynb), which is the two-stage model
this one abandons.


<!-- ML4T_ZH_BEGIN c0000-p001 -->
**檢查點時程分階段進行，且全部發布。** 訓練先執行無條件階段，再執行條件式階段；`case_studies/config/sdf/sdf.yaml` 宣告各階段內的儲存時點。預先固定最後一個檢查點，是看清問題的正確直覺：依最佳驗證 IC 選擇檢查點，就是看到答案後才做選擇。但只保留最後一個卻是錯誤補救方式，因為它丟掉了個案研究已花成本訓練的其他模型，並把選擇留給下游再次進行，卻沒有紀錄。發布完整時程，讓 [`14_backtest`](14_backtest.ipynb) 能在所有候選模型可見、規則明確的情況下，依驗證回測夏普比率選擇。**選擇在那裡進行，不在這裡。**

**學習目標。** 完成本 Notebook 後，你將能夠：

- 說明定價目標與預測目標相比，另外要求什麼。
- 解讀分階段訓練時程，並算出每個標籤應產生多少組預測。
- 從已解析規格解讀決策當時可得資料的約定，並說明它排除了什麼。
- 說明為何總體經濟背景資訊屬於模型識別資訊，而非附註。

**書籍參考**：第 14 章第 14.7 節（隨機折現因子與監督式自編碼器模型）。第 6 章第 6.7 節（搜尋紀錄與執行記錄）介紹本 Notebook 寫入的執行記錄。

**先備知識**：[`03_financial_features`](03_financial_features.ipynb) 與 [`04_model_based_features`](04_model_based_features.ipynb) 說明特徵矩陣；[`05_evaluation`](05_evaluation.ipynb) 說明滾動向前的各折；[`11c_conditional_autoencoder`](11c_conditional_autoencoder.ipynb) 則是此模型捨棄的兩階段形式。
<!-- ML4T_ZH_END c0000-p001 -->

**What it writes**: one training run per label and one complete validation prediction set per
label and checkpoint, in `run_log/registry.db` and under `run_log/training/` and
`run_log/predictions/`, grouped under a population named for this model. The family splits across
five notebooks, so each publishes its own population rather than one shared one.

<!-- ML4T_ZH_BEGIN c0000-p002 -->
**寫入內容**：每個標籤各有一筆訓練執行記錄，且每個標籤與檢查點各有一組完整的驗證預測，存於 `run_log/registry.db`，並位於 `run_log/training/` 與 `run_log/predictions/` 之下，歸入以此模型命名的模型群。這個家族分布於五份 Notebook，因此各自發布自己的模型群，而非共用一個。
<!-- ML4T_ZH_END c0000-p002 -->


In [ ]:
"""Fit the declared ETF stochastic-discount-factor population on the walk-forward folds."""

# Load PyTorch's bundled CUDA runtime before other ML4T libraries.
import torch  # noqa: F401

# isort: split
import plotly.graph_objects as go
import polars as pl
from plotly.subplots import make_subplots

from case_studies.research import (
    declared_labels,
    load_model_configs,
    model_requests,
    open_study,
    primary_label,
    resolved_model_plan,
    run_model_population,
)
from utils.style import COLORS, show_plotly_with_alt

In [ ]:
LABELS: list[str] = []
EXECUTION_TIER = "canonical"
WORKSPACE: str = ""
PREVIEW_REDUCTIONS: dict = {}
POPULATION_NAME = ""
SUPERSEDES_POPULATION: str = ""
DEVICE: str = ""

MODEL_NAME = "sdf"

In [ ]:
study = open_study("etfs", execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)

## 1. Which labels, and what the configuration says

Every label whose training menu declares `latent_factors:` is fitted, and both do: `fwd_ret_21d`,
the total return over the 21 trading days after the decision date, and `fwd_ret_5d`, the same
thing over five. The pricing objective is written against the label's returns, so these are two
separate fits rather than one fit scored twice. `LABELS` restricts the run to a subset when you
want one.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
## 1. 要使用哪些標籤，以及設定代表什麼

訓練選單中宣告 `latent_factors:` 的每個標籤都會擬合，兩個標籤皆符合：`fwd_ret_21d` 是決策日後 21 個交易日的總報酬率；`fwd_ret_5d` 則是後續五個交易日的同類報酬率。定價目標以標籤的報酬率表示，因此這是兩次獨立擬合，而非同一擬合評分兩次。需要時，`LABELS` 可將執行限制在子集。
<!-- ML4T_ZH_END c0004-p000 -->


In [ ]:
declared_labels(study, "latent_factors")

`case_studies/config/sdf/sdf.yaml` declares the estimator and its schedule, and the schedule has
more structure here than in the two autoencoders. `n_epochs_unc` trains the unconditional stage,
`n_epochs_cond` the conditional one that follows it, and `checkpoint_epochs` lists the saves
inside a stage. The resolved plan in section 2 turns those into the actual list of checkpoints,
which is what decides how many prediction sets each label owes - the count is not the length of
the declared list, because a save inside the unconditional stage and the same offset inside the
conditional one are two different states of the model.

<!-- ML4T_ZH_BEGIN c0006-p000 -->
`case_studies/config/sdf/sdf.yaml` 宣告估計器及其時程；此處的時程比另外兩個自編碼器更有結構。`n_epochs_unc` 訓練無條件階段，`n_epochs_cond` 訓練後續的條件式階段，而 `checkpoint_epochs` 列出階段內的儲存時點。第 2 節的已解析計畫將這些設定轉為實際的檢查點清單，並據此決定每個標籤應產生多少組預測；數量並非宣告清單的長度，因為無條件階段內的一次儲存，與條件式階段中相同偏移位置的儲存，是模型的兩種不同狀態。
<!-- ML4T_ZH_END c0006-p000 -->


In [ ]:
configs = load_model_configs(
    study,
    "latent_factors",
    labels=LABELS or None,
    config_names=[MODEL_NAME],
)
configs.select("label", "config_name", "params")

`LABELS` narrows what is fitted, and a narrowed run declares a different set of members than the
canonical population does. A population is immutable once written, so such a run must publish
under its own name: on a fresh workspace it would otherwise register an incomplete snapshot under
the canonical one, and where the full population already exists the registry refuses it.

The comparison is against this model's own declared rows rather than the whole `latent_factors`
catalog. The family is split across five notebooks and each publishes one model, so the complete
catalog is five times what this one publishes, and comparing against it would report every
canonical run as narrowed.

The device is checked in the same cell. Two networks trained on a GPU and the same two trained on
a CPU accumulate their sums in different orders and reach different weights, so the device is
part of what the fitted model is and is recorded inside the computation's identity rather than
beside it. `PUBLISHED_DEVICE` is the device this population was fitted on; the runner refuses to
substitute a CPU for a requested GPU rather than publishing a different model under the published
name, so on a machine with no NVIDIA card this notebook stops at the next cell. Set
`DEVICE="cpu"` and pass a `POPULATION_NAME` to fit the same schedule there.

<!-- ML4T_ZH_BEGIN c0008-p000 -->
`LABELS` 會縮小擬合範圍，而範圍縮小後宣告的成員集合，與標準模型群不同。模型群一旦寫入便不可變更，因此這類執行必須以自己的名稱發布：在全新工作區中，否則會以標準名稱登錄不完整的快照；若完整模型群已存在，登錄系統則會拒絕。

比較基準是此模型自行宣告的資料列，而非整個 `latent_factors` 目錄。此家族分布於五份 Notebook，每份各自發布一個模型；完整目錄的規模是此處發布內容的五倍。若拿完整目錄比較，所有標準執行都會被誤報為縮小範圍。

裝置也在同一個儲存格中檢查。兩個神經網路在 GPU 與 CPU 上訓練時，加總順序不同，最終權重也不同，因此裝置決定擬合模型的內容，會記入運算的識別資訊，而非僅作附註。`PUBLISHED_DEVICE` 是此模型群擬合所使用的裝置；執行器拒絕以 CPU 取代指定的 GPU，避免在已發布名稱下發布不同模型。所以，沒有 NVIDIA 顯示卡的機器會在下一個儲存格停止。若要在該機器擬合相同時程，請設定 `DEVICE="cpu"` 並傳入 `POPULATION_NAME`。
<!-- ML4T_ZH_END c0008-p000 -->


In [ ]:
PUBLISHED_DEVICE = "cuda"
device = DEVICE or PUBLISHED_DEVICE
print(f"training device: {device}")

declared = load_model_configs(study, "latent_factors", config_names=[MODEL_NAME])
narrowed = set(configs.get_column("label")) != set(declared.get_column("label"))
if (narrowed or device != PUBLISHED_DEVICE) and not POPULATION_NAME:
    raise ValueError(
        f"this run fits {configs.height} of the {declared.height} declared labels on device "
        f"{device!r}, which is not the complete declared catalog on {PUBLISHED_DEVICE!r}, so it "
        "cannot publish the canonical population; pass POPULATION_NAME to give it its own"
    )

## 2. Binding the declarations to the data

A menu entry says which estimator to fit. It does not say which fund-date pairs have both a
feature row and a label, or where the walk-forward folds fall. **Resolving** a request goes and
finds that: it reads the label, feature and macro files, computes the fold boundaries from the
walk-forward parameters in `config/setup.yaml`, and works out the exact set of rows each fit is
expected to predict. It fits nothing, so the plan can be read before any training starts.

Four things to check in it:

- **`feature_count`, `eligible_entities` and `eligible_rows` agree across the rows.** They are
  the width of the networks' input, the number of funds in the cross-section, and the fund-date
  pairs to be predicted. A row that differs is a label measured on a different sample.
- **`folds` is the same on both rows**, and equals the number of walk-forward splits
  [`05_evaluation`](05_evaluation.ipynb) established.
- **`validation_start` and `validation_end` bracket the development sample.** The held-out tail
  is scored once, at the end of the case study; any of it visible here would mean it had been
  used to choose something.
- **`checkpoints`** is the resolved schedule length, and it is the number to multiply by the row
  count to get the candidate models this notebook is about to create.

Each row also carries a `training_hash`: the identity of that computation, derived from
everything that can change its result. [`RUN_LOG.md`](../RUN_LOG.md#identity) sets out what goes
into one and what follows from it.

<!-- ML4T_ZH_BEGIN c0010-p000 -->
## 2. 將宣告內容與資料連結

選單項目指定要擬合哪一個估計器，卻不說明哪些基金與日期的配對同時具備特徵資料列和標籤，也不說明滾動向前的各折如何劃分。**解析**請求就是找出這些資訊：讀取標籤、特徵及總體經濟檔案，根據 `config/setup.yaml` 中的滾動向前參數計算各折邊界，並確定每次擬合應預測的確切資料列。這一步不會擬合模型，因此可在訓練開始前檢視計畫。

其中有四點需要檢查：

- **`feature_count`、`eligible_entities` 與 `eligible_rows` 在各列一致。** 它們分別是神經網路輸入的寬度、橫斷面的基金數，以及待預測的基金與日期配對。若某列不同，表示該標籤是在不同樣本上測得。
- **`folds` 在兩列中相同**，等於 [`05_evaluation`](05_evaluation.ipynb) 設定的滾動向前切分數。
- **`validation_start` 與 `validation_end` 界定開發樣本的起訖。** 保留的尾段只會在個案研究結尾評分一次；若其中任何資料在此可見，就代表曾用於選擇。
- **`checkpoints`** 是解析後的時程長度，將其乘以資料列數，就得到本 Notebook 即將建立的候選模型數。

每列還帶有 `training_hash`：這項運算的識別資訊，由所有可能改變結果的因素推導而來。[`RUN_LOG.md`](../RUN_LOG.md#identity) 說明其中包含什麼，以及由此產生的影響。
<!-- ML4T_ZH_END c0010-p000 -->


In [ ]:
requests = model_requests(
    study,
    configs,
    execution_tier=EXECUTION_TIER,
    overrides={"device": device},
    preview_reductions=PREVIEW_REDUCTIONS,
)
resolved = tuple(request.resolve() for request in requests)

plan = resolved_model_plan(resolved)
plan.select(
    "label",
    "config_name",
    "feature_count",
    "eligible_entities",
    "eligible_rows",
    "folds",
    "checkpoints",
    "validation_start",
    "validation_end",
)

The plan is also where a short population is visible. A run that fitted fewer labels than the
menu declares would still print an IC table and still register its rows; what it would not do is
announce the gap.

<!-- ML4T_ZH_BEGIN c0012-p000 -->
計畫也能顯示成員不足的模型群。若一次執行擬合的標籤比選單宣告的少，仍會列印 IC 表並登錄其資料列，卻不會主動指出缺漏。
<!-- ML4T_ZH_END c0012-p000 -->


In [ ]:
planned_pairs = set(zip(plan["label"], plan["config_name"], strict=True))
requested_pairs = set(zip(configs["label"], configs["config_name"], strict=True))
if planned_pairs != requested_pairs:
    raise RuntimeError(
        "the plan does not match the loaded SDF menu; "
        f"missing {sorted(requested_pairs - planned_pairs)}, "
        f"unexpected {sorted(planned_pairs - requested_pairs)}"
    )
print(f"{len(requested_pairs)} label-configuration pairs")

### The macro contract, read back from what was resolved

The discount factor takes macro context, and the point-in-time question that raises is not
rhetorical: a macro series as it reads today is not what it read on a decision date years ago,
because the agencies that publish these numbers revise them. Training on the revised series would
let the model condition on information nobody had.

The contract below is read out of the resolved specification rather than restated here, so it is
what actually entered the identity:

- **`source`** says the values are ALFRED *initial releases* - each series as it was first
  published, not as it now reads.
- **`availability_lag_days`** and **`alignment`** say a value dated *t* becomes usable on *t+1*
  and is then carried backward, so a same-day close cannot reach a same-day decision.
- **`coverage_start`** is where the panel begins, which is set by the last series to start rather
  than the first.
- **`resolved_fold_digest`** is the content of the macro values each fold actually saw. It is
  inside the training identity, so a macro panel that changed underneath would register as a
  different model rather than overwriting this one.

**What this does not buy.** The financial feature matrix still contains yield-curve variables
built from finalized FRED series, so the run as a whole is not a vintage-clean live estimate. The
macro contract covers the macro context and nothing else, and the feature matrix's exact content
is pinned by the modeling-input digest rather than by a release policy.

<!-- ML4T_ZH_BEGIN c0014-p000 -->
### 從已解析規格讀回總體經濟資料約定

折現因子使用總體經濟背景資訊，由此產生的「決策當時可得資訊」問題，並非修辭：今天讀取的總體經濟序列，與多年前某個決策日當時看到的數值不同，因為發布數據的機構會修訂它們。若使用修訂後序列訓練，就會讓模型依當時無人掌握的資訊調整。

下列約定從已解析的規格讀取，而非在此重述，因此它確實進入了識別資訊：

- **`source`** 表示數值為 ALFRED 的*首次發布值*，即各序列最初發布時的內容，而非目前讀到的修訂值。
- **`availability_lag_days`** 與 **`alignment`** 表示日期為 *t* 的數值從 *t+1* 才可使用，之後會向後沿用，因此當日收盤價不能進入當日決策。
- **`coverage_start`** 是面板資料的起點，由最晚開始的序列決定，而非最早開始的序列。
- **`resolved_fold_digest`** 是各折實際看到的總體經濟數值內容。它包含在訓練識別資訊內，因此若底層總體經濟面板變動，會登錄為不同模型，而非覆寫此模型。

**這項約定不能保證什麼。** 金融特徵矩陣仍包含根據最終版 FRED 序列建立的殖利率曲線變數，因此整次執行並非完全符合歷史版本資訊的即時估計。總體經濟資料約定只涵蓋總體經濟背景資訊；特徵矩陣的確切內容則由建模輸入的摘要值鎖定，而非由發布政策決定。
<!-- ML4T_ZH_END c0014-p000 -->


In [ ]:
macro = resolved[0].spec["computation"]["macro_context"]
if macro is None:
    raise RuntimeError("the SDF request resolved without a macro context")

pl.DataFrame(
    {
        "field": ["source", "policy", "version", "availability_lag_days", "alignment"]
        + ["coverage_start", "series", "resolved_fold_digest"],
        "value": [
            str(macro["source"]),
            str(macro["policy"]),
            str(macro["version"]),
            str(macro["availability_lag_days"]),
            str(macro["alignment"]),
            str(macro["coverage_start"]),
            str(len(macro["series"])),
            str(macro["resolved_fold_digest"]),
        ],
    }
)

## 3. Fitting the population

`run_model_population` runs every resolved request. For one request it walks the folds, and on
each one:

1. takes the fund-dates inside that fold's training window, together with the macro values that
   were available on those dates,
2. trains the unconditional stage, then the conditional one that adds the macro context, by
   gradient descent on the pricing errors, writing the weights to disk at each declared save,
3. applies each saved set of weights to the validation dates to get each fund's exposure to the
   discount factor there, and turns that into a predicted return.

Step 3 is what makes one training run produce several results. The fold predictions are
concatenated into one series per checkpoint covering the whole validation period, and each
becomes its own registered prediction set with its own identity.

**What the call publishes is a population**: a named, immutable list of the prediction sets it
will produce, computed from the resolved specifications and written down before the first fit.
Afterwards every member must exist and be complete, which is what makes the downstream comparison
well defined - [`14_backtest`](14_backtest.ipynb) backtests this population, not whatever
predictions happen to be in the registry - and it is why a configuration that raises fails the
whole call rather than publishing a population one member short.


<!-- ML4T_ZH_BEGIN c0016-p000 -->
## 3. 擬合模型群

`run_model_population` 執行每個已解析的請求。對單一請求，它依序處理各折；在每一折中：

1. 取出該折訓練期間內的基金與日期資料，以及當時可用的總體經濟數值；
2. 先訓練無條件階段，再以定價誤差為目標進行梯度下降，訓練加入總體經濟背景資訊的條件式階段，並在每個宣告的儲存時點將權重寫入磁碟；
3. 將每組已儲存的權重套用至驗證日期，取得各基金當時對折現因子的曝險，再轉成預測報酬率。

第 3 步使一次訓練產生多組結果。各折預測會串接成每個檢查點涵蓋整個驗證期間的一個序列，各自成為具備獨立識別資訊的已登錄預測集。

**此呼叫發布的是一個模型群**：預計產生的預測集所組成、具名且不可變更的清單；它根據已解析的規格計算，並在第一次擬合前寫下。之後每個成員都必須存在且完整，下游比較才有明確定義：[`14_backtest`](14_backtest.ipynb) 回測的是這個模型群，而非登錄系統中碰巧存在的預測。因此，若某設定拋出錯誤，整次呼叫都會失敗，不會發布缺少一個成員的模型群。
<!-- ML4T_ZH_END c0016-p000 -->

`SUPERSEDES_POPULATION` names the population hash this run replaces. A population is the set of
prediction identities, so anything that moves a training identity - a changed macro panel as
much as a changed epoch schedule - produces a different set under the same name, and the
registry refuses to write it without being told which snapshot it supersedes. It is empty here
because `etfs-sdf-validation-v1` has one generation and this run is it: there is no earlier
snapshot to replace. It stays empty on a re-run too, because reproducing a member list already
published under that name is not a replacement. Fill it in when you have changed something that
moves an identity and want the new set to take the name; the error raised on the attempt tells
you which hash to name. A reduced-scale run passes it empty whatever the default is: a
population produced under a reduction is thrown away with the workspace it was written to, so it
has no lineage to extend.

<!-- ML4T_ZH_BEGIN c0016-p001 -->
`SUPERSEDES_POPULATION` 指定本次執行要取代的模型群雜湊值。模型群是預測識別資訊的集合，因此任何改變訓練識別資訊的因素，無論是總體經濟面板還是訓練週期時程變動，都會在同一名稱下形成不同集合；若未指定被取代的快照，登錄系統會拒絕寫入。此處留空，因為 `etfs-sdf-validation-v1` 只有一代，而這次執行就是它：沒有可供取代的舊快照。重跑時也維持空值，因為重現已在該名稱下發布的成員清單不算取代。若你改變了會影響識別資訊的內容，且希望新集合沿用該名稱，便須填入此值；嘗試執行時的錯誤會告知應指定哪個雜湊值。縮小規模的執行無論預設值為何，都會傳入空值：縮小範圍產生的模型群會隨其工作區一起丟棄，沒有可延續的沿革。
<!-- ML4T_ZH_END c0016-p001 -->


In [ ]:
population_name = POPULATION_NAME or f"etfs-{MODEL_NAME}-validation-v1"
execution, population = run_model_population(
    study,
    resolved,
    population_name=population_name,
    supersedes=SUPERSEDES_POPULATION or None,
)

published_sets = sum(len(run.predictions) for run in execution.runs)
print(f"{len(execution.runs)} configurations, {published_sets} prediction sets")
print(f"population {population.name}: {len(population.members)} members")

A second run of this notebook trains nothing. Every identity is re-derived from the inputs, the
registry already holds the matching rows and the saved weights, and the runner returns the stored
result rather than training again - so re-running it unchanged costs the time it takes to read
the data. The latent-factor runner reports no per-fold fitted-or-reused breakdown, unlike the
linear and boosted families, so the counts above are of what the population holds rather than of
what this particular run computed.

### Running configurations of your own

The published run log is read-only. To add runs, open the study against a workspace, which holds
its own registry and artifacts and reads the same labels, features and macro panel:

```python
study = open_study("etfs", workspace="~/ml4t-experiments")
configs = load_model_configs(study, "latent_factors", labels=["fwd_ret_5d"], config_names=["sdf"])
requests = model_requests(study, configs, overrides={"device": "cuda"})
resolved = tuple(request.resolve() for request in requests)
execution, population = run_model_population(study, resolved, population_name="my-sdf-v1")
```

To change the architecture or the schedule, edit `case_studies/config/sdf/sdf.yaml`. That changes
this configuration's identity, so its result registers as a new row beside the old one rather
than replacing it - and that includes the phase lengths and `checkpoint_epochs`, which decide how
many members the population has. Give the run its own `population_name`: a name refers to one set
of members permanently, and reusing it for a different set raises.
[`RUN_LOG.md`](../RUN_LOG.md#running-your-own-configurations) covers the rest.

<!-- ML4T_ZH_BEGIN c0018-p000 -->
第二次執行本 Notebook 時不會訓練任何模型。所有識別資訊都從輸入重新推導；登錄系統已有相符資料列及儲存的權重，因此執行器會回傳儲存的結果，而非重新訓練。所以在內容不變時重跑，只需讀取資料的時間。與線性及提升式模型家族不同，潛在因子執行器不會分別報告各折是新擬合還是重用，因此上述計數代表模型群持有的內容，而非本次執行實際計算的內容。

### 執行自己的設定

已發布的執行記錄為唯讀。若要新增執行，請在工作區中開啟此研究；工作區有自己的登錄系統與成品，並讀取相同的標籤、特徵及總體經濟面板：

```python
study = open_study("etfs", workspace="~/ml4t-experiments")
configs = load_model_configs(study, "latent_factors", labels=["fwd_ret_5d"], config_names=["sdf"])
requests = model_requests(study, configs, overrides={"device": "cuda"})
resolved = tuple(request.resolve() for request in requests)
execution, population = run_model_population(study, resolved, population_name="my-sdf-v1")
```

若要變更架構或時程，請編輯 `case_studies/config/sdf/sdf.yaml`。這會改變此設定的識別資訊，因此結果會作為新資料列登錄在舊資料列旁，而非取代舊資料列；這也包括階段長度及決定模型群成員數的 `checkpoint_epochs`。請為這次執行指定自己的 `population_name`：一個名稱會永久對應一組成員，用於不同集合就會拋出錯誤。[`RUN_LOG.md`](../RUN_LOG.md#running-your-own-configurations) 說明其餘事項。
<!-- ML4T_ZH_END c0018-p000 -->


## 4. What came out

One row per label and checkpoint, read back from the registry. `ic_mean` is the **information
coefficient**: on each validation date, rank the funds by the model's prediction, rank them by
the return they went on to earn, correlate the two rankings, and average that daily correlation
over the validation period.

`ic_n_days` is how many validation dates produced a defined correlation, and it decides which
rows are comparable with each other. A model that has settled into assigning nearly the same
exposure to every fund on a date gives that date no spread to rank, and the date drops out of the
average - so that checkpoint's `ic_mean` is an average over a different sample from its
neighbours'. `full_coverage` marks the rows measured on every date their own label offers, and
the curves below are restricted to those. **Coverage is judged within a label**, because a
21-day forward return runs out of window earlier than a five-day one and one global maximum would
mark the whole 21-day grid incomplete for a reason that has nothing to do with the models.

The published catalog is checked against the population planned before fitting rather than
against its own row count, because a run that lost a member would otherwise report a shorter
table and nothing else.

<!-- ML4T_ZH_BEGIN c0019-p000 -->
## 4. 得到了什麼結果

從登錄系統讀回每個標籤與檢查點的一列。`ic_mean` 是**資訊係數**：在每個驗證日期，依模型預測為基金排序，再依基金後來實現的報酬率排序，計算兩組排名的相關性，最後對驗證期間的每日相關性取平均。

`ic_n_days` 是能算出相關性的驗證日期數，也決定哪些資料列能互相比較。若模型在某日對幾乎所有基金都給予相同曝險，當日便沒有可供排名的差異，會從平均值中剔除；因此該檢查點的 `ic_mean` 是在不同於相鄰檢查點的樣本上計算的平均值。`full_coverage` 標記涵蓋自身標籤所有可用日期的資料列，下方曲線只納入這些資料列。**涵蓋率須在同一標籤內判斷**，因為 21 日未來報酬率比五日報酬率更早超出視窗；若使用單一全域最大值，便會把整個 21 日網格標為不完整，原因卻與模型無關。

已發布的目錄會與擬合前規劃的模型群比較，而非只核對自身的資料列數；否則若執行時遺失一個成員，只會得到較短的表格，沒有其他提示。
<!-- ML4T_ZH_END c0019-p000 -->


In [ ]:
catalog = execution.catalog_rows.select(
    "label",
    "config_name",
    "complete",
    "checkpoint_kind",
    "checkpoint_value",
    "ic_mean",
    "ic_std",
    "ic_n_days",
    "n_folds",
    "training_hash",
    "prediction_hash",
).sort("label", "checkpoint_value")

if set(catalog.get_column("prediction_hash")) != set(population.members):
    raise RuntimeError("the published catalog differs from the population planned before fitting")
if catalog.filter(~pl.col("complete")).height:
    raise RuntimeError("a partial SDF prediction set cannot pass to backtesting")

catalog = catalog.with_columns(
    full_coverage=pl.col("ic_n_days") == pl.col("ic_n_days").max().over("label")
)

primary = primary_label(study)
present = sorted(set(catalog.get_column("label")))
# The primary label leads when it was fitted. A subset run that leaves it out orders the panels by
# whichever label it did fit rather than by one that is not there.
panel_labels = [label for label in [primary] if label in present] + [
    label for label in present if label != primary
]
print(f"{catalog.height} candidate models across {len(panel_labels)} labels")
print(f"at {catalog.n_unique('checkpoint_value')} checkpoints each")
catalog.select("label", "checkpoint_value", "ic_mean", "ic_std", "ic_n_days", "full_coverage")

### What more training does

Each line traces one label's out-of-sample IC across the phased schedule. The horizontal axis is
cumulative epochs, so the first point is the end of the unconditional stage and the rest are
saves inside the conditional one - the step from the first to the second is where the macro
context enters, and it is the only step on the axis that changes what the model is allowed to
see rather than only how long it has trained.

A line that rises and then falls has an interior optimum: the model was still learning, then
began fitting the training window at the expense of the validation folds. A line that wanders
around zero without trend never had anything to learn, and its highest point is wherever the
noise happened to peak. Both produce a respectable-looking maximum, which is why the maximum is
not what a checkpoint is judged on and why the whole schedule is published rather than the best
of it.

One panel per label, each with its own vertical scale, because a horizon with something to learn
and one without would be averaged into a single indistinct band if they shared axes.

<!-- ML4T_ZH_BEGIN c0021-p000 -->
### 增加訓練後會如何

每條線追蹤一個標籤的樣本外 IC，沿著分階段時程變動。橫軸是累計訓練週期，因此第一個點是無條件階段的終點，其餘是條件式階段內的儲存時點。從第一點到第二點時，總體經濟背景資訊開始進入模型；這是橫軸上唯一改變模型可見資訊的步驟，而不只是延長訓練時間。

先升後降的曲線在中途有最佳點：模型原本仍在學習，後來卻以犧牲驗證折表現為代價，開始過度擬合訓練期間。若曲線只是在零附近無趨勢地游移，表示模型從未有可學的訊號，最高點只是雜訊碰巧達峰。兩者都能產生看似不錯的最大值，因此不能只靠最大值評判檢查點，也因此發布完整時程，而非只發布最佳者。

每個標籤各有一幅圖，且各自使用縱軸刻度；若共用刻度，有可學訊號的預測期間與沒有訊號的預測期間會被壓成一條模糊的帶狀區域。
<!-- ML4T_ZH_END c0021-p000 -->


In [ ]:
curves = catalog.filter("full_coverage").sort("label", "checkpoint_value")
fig = make_subplots(
    rows=len(panel_labels),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=[
        f"{label} ({'primary' if label == primary else 'variant'})" for label in panel_labels
    ],
)
for row, label in enumerate(panel_labels, start=1):
    series = curves.filter(pl.col("label") == label)
    fig.add_trace(
        go.Scatter(
            x=series.get_column("checkpoint_value").to_list(),
            y=series.get_column("ic_mean").to_list(),
            mode="lines+markers",
            name=label,
            showlegend=False,
            line=dict(color=COLORS["blue"], width=2),
            marker=dict(size=6, color=COLORS["blue"]),
        ),
        row=row,
        col=1,
    )
    fig.add_hline(y=0, line_width=1, line_dash="dash", line_color=COLORS["neutral"], row=row, col=1)
    fig.update_yaxes(title_text="Mean IC (validation)", row=row, col=1)
fig.update_xaxes(title_text="Cumulative training epoch", row=len(panel_labels), col=1)
fig.update_layout(
    title="Out-of-sample ranking accuracy across the phased schedule",
    height=320 * len(panel_labels),
    width=800,
    margin=dict(t=90),
)
# Read off the frame rather than asserted, so the alt text stays true of the next run.
_spans = "; ".join(
    f"{label} from {curves.filter(pl.col('label') == label).get_column('ic_mean').min():.3f} "
    f"to {curves.filter(pl.col('label') == label).get_column('ic_mean').max():.3f}"
    for label in panel_labels
)
show_plotly_with_alt(
    fig,
    "Line chart of mean validation information coefficient against cumulative training epoch, one "
    "panel per label, each with its own vertical scale and a dashed zero line. Counted from the "
    f"frame: {_spans}.",
)

### What one checkpoint's number averages

Each row of the catalog is one number over the whole validation period, and the eight folds
behind it are trained on different windows and scored on different years. The table below
recomputes each fold's mean IC from the published predictions of one checkpoint, so the
disagreement between folds measuring the same model can be compared against the differences
between checkpoints in the curve above.

The checkpoint shown is the last of the schedule. That is where training stopped, not a
selection: reading these eight numbers at the checkpoint with the highest `ic_mean` and reporting
them as the model's would be the choice this notebook publishes the whole schedule to avoid.

<!-- ML4T_ZH_BEGIN c0023-p000 -->
### 一個檢查點的數值平均了什麼

目錄中每列都是整個驗證期間的一個數值，背後八折使用不同期間訓練，也在不同年度評分。下表依某個檢查點已發布的預測，重新計算各折的平均 IC，讓同一模型各折間的差異，能與上方曲線中各檢查點之間的差異比較。

這裡顯示的是時程中的最後一個檢查點。那是訓練停止的地方，而非一次選擇：若改在 `ic_mean` 最高的檢查點讀取這八個數字，再把它們報告為模型結果，便是在做本 Notebook 藉由發布完整時程來避免的選擇。
<!-- ML4T_ZH_END c0023-p000 -->


In [ ]:
published = {result.hash: result for run in execution.runs for result in run.predictions}
last_checkpoint = catalog.get_column("checkpoint_value").max()


def _fold_ic(prediction_hash: str) -> pl.DataFrame:
    """Mean daily rank correlation of prediction against realised return, within each fold."""
    return (
        published[prediction_hash]
        .load()
        .group_by("fold", "timestamp")
        .agg(pl.corr("prediction", "actual", method="spearman").alias("ic"))
        .group_by("fold")
        .agg(
            pl.col("ic").mean().alias("fold_ic"),
            pl.len().alias("validation_dates"),
        )
        .sort("fold")
    )


by_fold = {
    row["label"]: _fold_ic(row["prediction_hash"])
    for row in catalog.filter(pl.col("checkpoint_value") == last_checkpoint).iter_rows(named=True)
}
for label in panel_labels:
    ic = by_fold[label].get_column("fold_ic")
    # See 11b: polars returns None for the mean of an empty column and for the standard
    # deviation of a single row, and a narrowed population produces both.
    if ic.len() == 0:
        print(f"{label} at cumulative epoch {last_checkpoint}: no fold scored, so there is no IC")
        continue
    spread = (
        f"standard deviation across folds {ic.std():.4f}"
        if ic.len() > 1
        else "one fold only, so no spread across folds"
    )
    print(
        f"{label} at cumulative epoch {last_checkpoint}: mean {ic.mean():+.4f}, "
        f"{spread}, "
        f"{(ic < 0).sum()} of {ic.len()} folds negative"
    )

by_fold[panel_labels[0]]

## 5. What to notice

**The macro step is the whole story on the primary label, and it runs the other way on the
five-day one.** Checkpoint 256 is where the unconditional stage ends, and on `fwd_ret_21d` it
scores **-0.0070** - the model orders the cross-section slightly worse than a coin. The four
conditional saves that follow run **+0.0132, +0.0223, +0.0379, +0.0370**. Every bit of ranking
accuracy this notebook publishes on its primary label appears after the macro context enters, at
the one step on the axis that changes what the model is allowed to see rather than only how long
it has trained. On `fwd_ret_5d` the same step is a loss: 256 scores **+0.0394**, the highest of
its five, and the four conditional checkpoints sit **below** it at +0.0091, +0.0152, +0.0233 and
+0.0238. Conditioning on eleven macro series helps the 21-day horizon and hurts the 5-day one,
which is not a result that survives being averaged into one number for the model.

**Pricing the cross-section did not beat splitting the problem in two.** Against the members
fitted on the same panel over the same folds, this one's best primary-label checkpoint
(**+0.0379**) sits below [`11b_ipca`](11b_ipca.ipynb)'s single **+0.0413** and well below
[`11c_conditional_autoencoder`](11c_conditional_autoencoder.ipynb)'s **+0.0722** at the end of its
schedule. On the five-day label its best (+0.0394, from the unconditional stage) is near IPCA's
+0.0349 and under the autoencoder's +0.0457. It clears [`11a_pca`](11a_pca.ipynb)'s -0.0326 on
the primary label, so conditioning still bought something over conditioning on nothing - just
less than either two-stage member bought, at four times the training. That is the trade this
member exists to price, and here the answer is negative.


<!-- ML4T_ZH_BEGIN c0025-p000 -->
## 5. 值得注意的結果

**對主要標籤而言，加入總體經濟資料是整個結果的關鍵；對五日標籤卻產生相反效果。** 檢查點 256 是無條件階段的終點，在 `fwd_ret_21d` 上得到 **-0.0070**，模型對橫斷面的排序略差於擲硬幣。後續四個條件式儲存點依序為 **+0.0132、+0.0223、+0.0379、+0.0370**。本 Notebook 在主要標籤上發布的全部排序準確度，都出現在總體經濟背景資訊進入後；這是橫軸上唯一改變模型可見資訊的步驟，而不只是延長訓練時間。對 `fwd_ret_5d` 而言，同一步驟卻造成損失：檢查點 256 得分 **+0.0394**，是五個檢查點中的最高值；後續四個條件式檢查點均**低於**它，分別為 +0.0091、+0.0152、+0.0233 及 +0.0238。依十一個總體經濟序列調整，有助於 21 日預測期間，卻傷害五日預測期間；若把整個模型平均成單一數值，這項結果便會消失。

**直接為橫斷面定價，未能勝過將問題拆成兩部分。** 與在相同面板及各折上擬合的成員相比，此模型主要標籤的最佳檢查點（**+0.0379**）低於 [`11b_ipca`](11b_ipca.ipynb) 單一模型的 **+0.0413**，也遠低於 [`11c_conditional_autoencoder`](11c_conditional_autoencoder.ipynb) 在時程終點的 **+0.0722**。五日標籤的最佳值（無條件階段的 +0.0394）接近 IPCA 的 +0.0349，卻低於自編碼器的 +0.0457。它在主要標籤上超過 [`11a_pca`](11a_pca.ipynb) 的 -0.0326，顯示依資訊調整仍比完全不調整有價值；只是其提升小於另外兩個兩階段成員，訓練時間卻是四倍。衡量這項取捨正是此成員存在的目的，而此處的答案是否定的。
<!-- ML4T_ZH_END c0025-p000 -->

**The checkpoint curve is not evidence for an epoch, and the arithmetic says so plainly.** The
five primary-label checkpoints span 0.0449 of IC end to end. The eight fold ICs behind the last of
them - +0.005, +0.027, +0.165, +0.159, +0.077, +0.021, -0.140, -0.017 - have a standard deviation
across folds of **0.0989**, more than twice that span. The apparent best checkpoint, 1024, beats
1280 by 0.0009, a fiftieth of the disagreement between folds measuring the same model. Nothing
here distinguishes an epoch worth stopping at from an epoch that happened to score well on eight
folds, which is why all five are published and why nothing downstream may select among them on
validation IC. The dispersion is also no better than the neighbours': 0.0989 here against 0.092
for the conditional autoencoder and 0.099 for IPCA.

**No checkpoint collapsed, and that had to be checked rather than assumed.** `ic_n_days` is 1,995
on every primary-label row and 2,011 on every five-day row - the full count each label offers - so
`full_coverage` holds for all ten published models. A model that had settled into pricing every
fund alike on a date would have dropped that date out of its average and left one row measured on
a different sample from its neighbours'.


<!-- ML4T_ZH_BEGIN c0025-p001 -->
**檢查點曲線不能作為選擇某個訓練週期的證據，數字也清楚說明了這點。** 主要標籤的五個檢查點，IC 最大與最小相差 0.0449。最後一個檢查點背後的八折 IC 為 +0.005、+0.027、+0.165、+0.159、+0.077、+0.021、-0.140 及 -0.017；折間標準差為 **0.0989**，超過該跨度的兩倍。表面上的最佳檢查點 1024，比 1280 高出 0.0009，僅為同一模型折間分歧的五十分之一。此處無法區分值得停止的週期與只是在八折上碰巧得分較高的週期；因此發布全部五個檢查點，下游也不得依驗證 IC 在其中選擇。離散程度也未優於相鄰模型：此處是 0.0989，條件式自編碼器為 0.092，IPCA 為 0.099。

**沒有任何檢查點發生預測塌縮，而這必須檢查，不能直接假設。** 主要標籤每列的 `ic_n_days` 都是 1,995，五日標籤每列都是 2,011，均為各標籤可提供的完整日期數，因此所有十個已發布模型都符合 `full_coverage`。若某個模型在某日對所有基金幾乎給予相同定價，當日就會從平均值中剔除，使該列與相鄰資料列使用不同樣本。
<!-- ML4T_ZH_END c0025-p001 -->

**Known limitations.** The architecture, the factor count and the two-phase schedule are declared
in `case_studies/config/sdf/sdf.yaml` rather than searched, so nothing here says this is the right
SDF - only what this one did. The macro contract is point-in-time by construction, and its
coverage starts **2010-11-23** while the panel starts in 2006, so the earliest training windows
carry less macro history than the later ones and the conditional stage is not learning from the
same amount of context in every fold. Two of eight folds are negative on the primary label and
three on the five-day. The panel narrows from 96 funds in the first fold to 90 in the last. The
published result is a CUDA result, and a reader who refits on CPU gets a different identity
registered beside it rather than these numbers. And every number here is measured on validation
folds that have been read many times over by the time a case study reaches this notebook.

<!-- ML4T_ZH_BEGIN c0025-p002 -->
**已知限制。** 架構、因子數及兩階段時程都在 `case_studies/config/sdf/sdf.yaml` 中宣告，而非透過搜尋決定。因此，此處只能說明這個隨機折現因子的表現，無法證明它是正確選擇。總體經濟資料約定在設計上遵循決策當時可得的資訊，但涵蓋範圍從 **2010-11-23** 才開始，面板卻從 2006 年開始。因此，最早的訓練期間所含總體經濟歷史資料比後續期間少，條件式階段在各折並非依同等數量的背景資訊學習。主要標籤八折中有兩折為負，五日標籤則有三折。面板中的基金數從第一折的 96 檔縮至最後一折的 90 檔。已發布結果是在 CUDA 上取得；若讀者改在 CPU 上重新擬合，會以不同識別資訊登錄在旁邊，而非得到這組數字。此外，當個案研究進行到本 Notebook 時，這裡的每個數字都來自已被反覆檢視多次的驗證折。
<!-- ML4T_ZH_END c0025-p002 -->


**Next**: [`11e_supervised_autoencoder`](11e_supervised_autoencoder.ipynb) is the family's own
control - it keeps [`11c`](11c_conditional_autoencoder.ipynb)'s network and bottleneck while
dropping the factor interpretation, so what it does not achieve is what the structure was worth.
[`13_model_analysis`](13_model_analysis.ipynb) is where all five are read against the other
families.

<!-- ML4T_ZH_BEGIN c0026-p000 -->
**下一步**：[`11e_supervised_autoencoder`](11e_supervised_autoencoder.ipynb) 是此家族自身的對照模型：它保留 [`11c`](11c_conditional_autoencoder.ipynb) 的神經網路與瓶頸層，但捨棄因子解釋，因此其未能達成的效果，可以用來衡量因子結構的價值。[`13_model_analysis`](13_model_analysis.ipynb) 將這五個模型與其他家族一起比較。
<!-- ML4T_ZH_END c0026-p000 -->
